# DynaSurvCausalOnline — Identifiability-Consistent Model Validation

Corrected, identifiability-aware evaluation. Supersedes `model_validation.ipynb`,
which built the datamodule **without** the identifiability controls the current
checkpoints are trained with (calendar covariate, cohort-start restriction,
temporal holdout, excluded arms) and therefore fed the model silently
mis-composed features (`x_input_dim` off by one) — producing wrong numbers that
never raised an error.

**All configuration lives in the CONFIG cell below.** Everything downstream is
derived from those flags and from the project TOML configs, so the evaluation
stays consistent with training *by construction*. A feature-width assertion after
the datamodule is built fails loudly if the flags ever drift from the checkpoint.

## Objectives
1. **Factual consistency** — calibration and discrimination (C-index, Brier score).
2. **Counterfactual stability** — treatment-wise calibration.
3. **Clinical plausibility** — KM vs predicted survival.
4. **Policy evaluation** — model (RMST-optimal) recommendation vs observed treatment.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import Counter, defaultdict
from numbers import Number
from pathlib import Path
import tomllib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from lifelines import KaplanMeierFitter
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sksurv.nonparametric import kaplan_meier_estimator
from tqdm import tqdm

from CausalSurv.data.datamodule_cv import ESMEOnlineDataModuleCV
from CausalSurv.evaluation.evaluator import DynasurvEvaluator
from CausalSurv.model import DynaSurvCausalOnline

sns.set_theme(style="whitegrid")
%matplotlib inline

## Configuration

The only cell you should normally edit. The identifiability controls are read from
`configs/config.toml` / `configs/optuna_best.toml` so they match training; override
them here **only** if your checkpoint was trained with different values.

In [ ]:
# ============================================================================
#  CONFIGURATION
# ============================================================================

# --- Repo + config sources (single source of truth: keeps eval == train) ---
REPO_ROOT = Path("/Users/malek/TheLAB/DynaSurv")
DATA_CONFIG_PATH = REPO_ROOT / "configs" / "config.toml"
MODEL_CONFIG_PATH = REPO_ROOT / "configs" / "optuna_best.toml"

# --- Which run to evaluate --------------------------------------------------
SUBTYPE = "HR+HER2-"          # HR+HER2- | HER2+ | TN
N_LINES = 4
# The split seed fixes the temporal holdout and is baked into the checkpoint.
# It is the number in the run-folder name:  {date}_seed_{SPLIT_SEED}
SPLIT_SEED = 1237297231
# Which checkpoint of that run: "bestCI" | "bestIBS" | "last"
# or an explicit path to a .ckpt file.
CHECKPOINT = "bestCI"

# --- Evaluation knobs -------------------------------------------------------
BATCH_SIZE = 256
NUM_WORKERS = 4               # set to 0 if the dataloader misbehaves headless
PLOTS = True
EVAL_HORIZONS = None          # None -> [eval].horizon_times from config.toml
LINE_CALIB_TIMES = [6, 12, 18, 24]
TREATMENT_CALIB_TIMES = [6, 12, 18]
TRIAL_HORIZON = 10            # RMST horizon (months) for the virtual trial

# --- Identifiability controls (loaded from config; must match the checkpoint)
with open(DATA_CONFIG_PATH, "rb") as f:
    _cfg = tomllib.load(f)
DATA_CFG, EVAL_CFG = _cfg["data"], _cfg["eval"]
with open(MODEL_CONFIG_PATH, "rb") as f:
    MODEL_CFG = tomllib.load(f)

N_INTERVALS = MODEL_CFG["n_intervals"]
COHORT_START_YEAR = DATA_CFG.get("cohort_start_year")
TEMPORAL_SPLIT_YEAR = DATA_CFG.get("temporal_split_year")
ADD_CALENDAR_FEATURE = DATA_CFG.get("add_calendar_feature", False)
EXCLUDED_TREATMENT_ARMS = DATA_CFG.get("excluded_treatment_arms")
MIN_SAMPLES_PER_TREATMENT = DATA_CFG.get("min_samples_per_treatment", 200)
if EVAL_HORIZONS is None:
    EVAL_HORIZONS = EVAL_CFG["horizon_times"]

print("Identifiability config (must match the checkpoint):")
print(f"  n_intervals             = {N_INTERVALS}")
print(f"  add_calendar_feature    = {ADD_CALENDAR_FEATURE}")
print(f"  cohort_start_year       = {COHORT_START_YEAR}")
print(f"  temporal_split_year     = {TEMPORAL_SPLIT_YEAR}")
print(f"  excluded_treatment_arms = {EXCLUDED_TREATMENT_ARMS}")
print(f"  min_samples_per_treat.  = {MIN_SAMPLES_PER_TREATMENT}")
print(f"  eval horizons           = {EVAL_HORIZONS}")

In [ ]:
def resolve_checkpoint(subtype, n_lines, seed, which):
    """Locate a run's checkpoint file, or accept an explicit .ckpt path."""
    p = Path(which)
    if p.suffix == ".ckpt":
        return p
    base = REPO_ROOT / "models" / subtype / f"{n_lines}lines"
    runs = sorted(base.glob(f"*_seed_{seed}/checkpoints"))
    if not runs:
        raise FileNotFoundError(f"No run folder matching seed {seed} under {base}")
    run = runs[-1]
    pattern = {
        "bestCI": "*bestCI*.ckpt",
        "bestIBS": "*bestIBS*.ckpt",
        "last": "last.ckpt",
    }.get(which, f"*{which}*.ckpt")
    matches = sorted(run.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No checkpoint matching '{which}' in {run}")
    return matches[-1]


MODEL_PATH = resolve_checkpoint(SUBTYPE, N_LINES, SPLIT_SEED, CHECKPOINT)
print("Using checkpoint:", MODEL_PATH)

## Load model and data

In [ ]:
print(f"Loading model from {MODEL_PATH} ...")
model = DynaSurvCausalOnline.load_from_checkpoint(
    str(MODEL_PATH), map_location=torch.device("cpu")
)
model.eval()
model.freeze()
print(f"Model loaded. x_input_dim={model.x_input_dim}, "
      f"p_input_dim={model.p_input_dim}, "
      f"interval_bounds={tuple(model.interval_bounds.shape)}")

print("Building datamodule (identifiability controls from config.toml) ...")
data_module = ESMEOnlineDataModuleCV(
    data_dir=str(REPO_ROOT / "data"),
    subtype=SUBTYPE,
    n_lines=N_LINES,
    n_intervals=N_INTERVALS,
    batch_size=BATCH_SIZE,
    split_seed=SPLIT_SEED,
    final_training=True,
    num_workers=NUM_WORKERS,
    cohort_start_year=COHORT_START_YEAR,
    temporal_split_year=TEMPORAL_SPLIT_YEAR,
    add_calendar_feature=ADD_CALENDAR_FEATURE,
    excluded_treatment_arms=EXCLUDED_TREATMENT_ARMS,
    min_samples_per_treatment=MIN_SAMPLES_PER_TREATMENT,
)
data_module.prepare_data()
data_module.setup()

# Feature-width guard: the datamodule must produce exactly the width the
# checkpoint was trained on (XPd = x_input_dim + p_input_dim + buffer). A
# mismatch here is the silent-corruption bug the old notebook shipped with.
produced = data_module.ESMEDataset[0][0].shape[-1]
expected = model.x_input_dim + model.p_input_dim + 1
assert produced == expected, (
    f"Feature mismatch: datamodule produced XPd width {produced} but the "
    f"checkpoint expects {expected} (x={model.x_input_dim} + p={model.p_input_dim} "
    f"+ 1). The identifiability flags must match training — most often this is "
    f"ADD_CALENDAR_FEATURE."
)
print(f"Feature-width check OK (XPd width={produced}).")

evaluator = DynasurvEvaluator(model, data_module)
interval_bounds = model.interval_bounds.to("cpu")
test_loader = data_module.test_dataloader()
print("Evaluator ready.")

## Treatment distribution (restricted cohort)

In [ ]:
df = data_module.df_dynamic
df_counts = (
    df.loc[df["lineid"] <= N_LINES]
    .groupby(["lineid", "T_treatment_category"])
    .size()
    .reset_index(name="count")
)
pivot_df = df_counts.pivot(
    index="lineid", columns="T_treatment_category", values="count"
).fillna(0)
pivot_df.plot(
    kind="bar", stacked=True, width=0.75, edgecolor="white", linewidth=0.5
)
plt.title("Treatment distribution per line (cohort used for training)")
plt.ylabel("Patients")
plt.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
plt.tight_layout()
plt.show()
pivot_df

## Factual consistency check

In [ ]:
res = evaluator.test_model()

## Predicted survival — sample patients

In [ ]:
def plot_predicted_survival_sample(test_dataloader, n_samples, ncols=5):
    (XPd, X_static, interval_idx, treatment_indices,
     time, event, mask, patient_id) = next(iter(test_dataloader))

    rng = np.random.default_rng(0)
    sample_idx = rng.choice(np.arange(XPd.shape[0]), n_samples, replace=False)

    nrows = (n_samples + ncols - 1) // ncols
    fig, ax = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows),
                           squeeze=False)
    surv = model.predict_discrete_survival(
        XPd, X_static, gather=True, factual_idx=treatment_indices
    ).numpy()
    n_lines = surv.shape[1]
    clr = plt.colormaps["tab10"].colors[:n_lines]

    for c, pat_idx in enumerate(sample_idx):
        a = ax[c // ncols, c % ncols]
        for line in range(n_lines):
            if not bool(mask[pat_idx, line]):
                continue
            mask_line = mask[:, line].bool()
            true_t, true_p = kaplan_meier_estimator(
                event.squeeze()[mask_line, line].bool(),
                time.squeeze()[mask_line, line],
            )
            a.step(interval_bounds, surv[pat_idx, line],
                   label=f"line {line + 1}", color=clr[line])
            a.step(true_t, true_p, label=f"line {line + 1} KM",
                   color=clr[line], linestyle="--")
        a.set_title(f"Patient {int(patient_id[pat_idx])}")
        a.set_xlim(0, 50)
        a.legend(fontsize=7)
    plt.suptitle("Sample survival predictions vs KM")
    plt.tight_layout()
    plt.show()


plot_predicted_survival_sample(data_module.test_dataloader(), 5)

In [ ]:
def plot_times_distribution(test_dataloader):
    (XPd, *_, time, _, mask, _) = next(iter(test_dataloader))
    time, mask = time.numpy(), mask.numpy()
    n_lines = XPd.shape[1]
    fig, ax = plt.subplots(nrows=1, ncols=n_lines, figsize=(4 * n_lines, 3))
    for line in range(n_lines):
        valid = mask[:, line] == 1
        if not valid.any():
            continue
        t = time[valid, line]
        med, q90 = np.percentile(t, [50, 90])
        ax[line].hist(t, bins=50)
        ax[line].axvline(med, color="green", lw=2, label=f"Median = {med:.1f}")
        ax[line].axvline(q90, color="red", lw=2, label=f"90% = {q90:.1f}")
        ax[line].set_title(f"Line {line + 1}")
        ax[line].set_xlabel("Time")
        ax[line].legend(fontsize=8)
    fig.suptitle("Distribution of event times by treatment line")
    plt.tight_layout()
    plt.show()


plot_times_distribution(test_loader)

In [ ]:
def plot_km_per_line(model, test_dataloader):
    (XPd, X_static, interval_idx, treatment_indices,
     time, event, mask, patient_id) = next(iter(test_dataloader))
    n_lines = XPd.shape[1]
    fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(15, 3))
    for line in range(n_lines):
        valid = mask[:, line] == 1
        e_line = np.asarray(model.train_events[line]).ravel()
        t_line = np.asarray(model.train_times[line]).ravel()

        s_t, s_p = kaplan_meier_estimator(
            event.squeeze()[valid, line].bool(), time.squeeze()[valid, line]
        )
        tr_t, tr_p = kaplan_meier_estimator(e_line.astype(bool), t_line)
        cn_t, cn_p = kaplan_meier_estimator(~e_line.astype(bool), t_line)

        ax[0].step(s_t, s_p, label=f"Line {line + 1}")
        ax[1].step(tr_t, tr_p, label=f"Line {line + 1}")
        ax[2].step(cn_t, cn_p, label=f"Line {line + 1}")
    for a, title in zip(ax, ["test survival", "train survival", "train censoring"]):
        a.set_title(title)
        a.legend()
    plt.suptitle("Kaplan-Meier by line")
    plt.tight_layout()
    plt.show()


plot_km_per_line(model, data_module.test_dataloader())

## Brier score

In [ ]:
ibs_by_line, bs_by_line, ipcw = evaluator.brier_score(EVAL_HORIZONS, plot=PLOTS)

## Calibration

### Overall (per-line) calibration

In [ ]:
line_ece = evaluator.line_calibration_error(eval_time=LINE_CALIB_TIMES, plot=PLOTS)

### Treatment-wise calibration

In [ ]:
evaluator.treatment_calibration_error(eval_time=TREATMENT_CALIB_TIMES, plot=PLOTS)

### Predicted survival vs KM, per treatment arm

In [ ]:
evaluator.treatment_calibration_KM()

### Average predicted survival vs KM

In [ ]:
evaluator.average_km()

## Adversarial unconfounding

If the latent representation still carries treatment information, a classifier
trained on it will beat the stratified-dummy baseline. Lower AUC (closer to the
dummy) is better balance.

In [ ]:
XPd, X_static, interval_idx, treatment_indices, time, event, mask, patient_id = (
    next(iter(test_loader))
)
model.compute_treatment_prediction_auc(XPd, X_static, treatment_indices, mask)

In [ ]:
accuracies, auc_scores, dummy_auc_scores = {}, {}, {}

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Probing latent state"):
        (XPd, X_static, interval_idx, treatment_indices,
         time, event, mask, patient_id) = batch
        h, c, p = model._init_lstm_states(X_static, XPd.device)

        for line in range(XPd.shape[1]):
            # NOTE: the custom LSTM requires the per-line treatment index (it
            # conditions the gates on the treatment embedding). Omitting it — as
            # the old notebook did — raises a TypeError against the current API.
            _, h, c, p = model.lstm(
                XPd[:, line, :], (h, c, p), treatment_indices[:, line]
            )

            mask_line = mask[:, line].bool()
            if not mask_line.any():
                continue

            X = h[mask_line].cpu().numpy()
            y = treatment_indices[mask_line, line].cpu().numpy()

            classes, counts = np.unique(y, return_counts=True)
            keep = np.isin(y, classes[counts >= 5])
            X, y = X[keep], y[keep]
            if len(np.unique(y)) < 2:
                continue

            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=y
            )
            rfc = RandomForestClassifier(n_estimators=100, random_state=42)
            rfc.fit(X_tr, y_tr)
            dummy = DummyClassifier(strategy="stratified", random_state=42)
            dummy.fit(X_tr, y_tr)

            accuracies[line] = rfc.score(X_te, y_te)
            auc_scores[line] = roc_auc_score(
                y_te, rfc.predict_proba(X_te), multi_class="ovr",
                average="macro", labels=rfc.classes_,
            )
            dummy_auc_scores[line] = roc_auc_score(
                y_te, dummy.predict_proba(X_te), multi_class="ovr",
                average="macro", labels=dummy.classes_,
            )

print(f"\n{'Line':<8} | {'RFC acc':<9} | {'RFC AUC':<9} | {'Dummy AUC':<9}")
print("-" * 46)
for line in range(N_LINES):
    if line in accuracies:
        print(f"Line {line + 1:<3} | {accuracies[line]:<9.4f} | "
              f"{auc_scores[line]:<9.4f} | {dummy_auc_scores[line]:<9.4f}")
    else:
        print(f"Line {line + 1:<3} | (no valid data)")

## Virtual clinical trial

For each line the model ranks arms by RMST at `TRIAL_HORIZON` and picks the best.
Patients whose observed arm matches the recommendation ("matched") are compared to
those whose arm differed ("differed") by KM and RMST(t).

In [ ]:
def run_ai_vs_doctor(dataloader, interval_bounds, eval_time=50):
    """Recommend the RMST-optimal arm per line and split patients by whether the
    observed arm matched the recommendation."""
    (XPd, X_static, interval_idx, treatment_indices,
     time, event, mask, patient_id) = next(iter(dataloader))
    batch_size, n_lines = XPd.shape[:2]
    n_treatments = model.n_treatments

    if isinstance(eval_time, Number):
        eval_time = [eval_time] * n_lines
    assert isinstance(eval_time, list)

    concordant = defaultdict(lambda: defaultdict(list))
    discordant = defaultdict(lambda: defaultdict(list))

    # Counterfactual survival for every arm (gather=False), with the encoder
    # still conditioned on the factual sequence via factual_idx (required).
    disc_survival = model.predict_discrete_survival(
        XPd, X_static, gather=False, factual_idx=treatment_indices
    )  # (batch, n_lines, n_treatments, n_intervals + 1)
    dt = interval_bounds[1] - interval_bounds[0]

    rmst = torch.cumsum(disc_survival, dim=3) * dt
    rmst_idx = torch.bucketize(
        torch.tensor(eval_time, dtype=interval_bounds.dtype).view(n_lines, -1),
        interval_bounds,
    )  # (n_lines, 1)
    gather_idx = rmst_idx.unsqueeze(0).unsqueeze(-1).expand(
        batch_size, n_lines, n_treatments, -1
    )
    rmst_at_h = torch.gather(rmst, dim=3, index=gather_idx).squeeze(-1)

    recommended = rmst_at_h.argmax(dim=2)  # (batch, n_lines)
    observed = treatment_indices

    for line in range(n_lines):
        valid = mask[:, line].bool()
        if not valid.any():
            continue
        rec = recommended[valid, line].cpu().numpy()
        obs = observed[valid, line].cpu().numpy()
        t = time[valid, line].cpu().numpy()
        e = event[valid, line].cpu().numpy()
        matched = rec == obs
        concordant[line]["time"].extend(t[matched])
        concordant[line]["event"].extend(e[matched])
        discordant[line]["time"].extend(t[~matched])
        discordant[line]["event"].extend(e[~matched])

    return concordant, discordant, recommended, observed, mask

In [ ]:
def _rmst_from_km(kmf):
    t = kmf.survival_function_.index.values
    s = kmf.survival_function_[kmf.label].values
    dt = np.diff(np.insert(t, 0, 0))
    return t, np.cumsum(s * dt)


def plot_conc_disc(concordant, discordant, xmax=50):
    n_lines = len(concordant)
    fig, axes = plt.subplots(2, n_lines, figsize=(6 * n_lines, 8), sharex="col")
    axes = np.atleast_2d(axes)
    if axes.shape[0] == 1:
        axes = axes.reshape(2, -1)

    for line in range(n_lines):
        ax_km, ax_rmst = axes[0, line], axes[1, line]
        kmf_c, kmf_d = KaplanMeierFitter(), KaplanMeierFitter()
        Tc, Ec = concordant[line]["time"], concordant[line]["event"]
        Td, Ed = discordant[line]["time"], discordant[line]["event"]

        kmf_c.fit(Tc, event_observed=Ec, label=f"Matched (n={len(Tc)})")
        kmf_d.fit(Td, event_observed=Ed, label=f"Differed (n={len(Td)})")
        kmf_c.plot_survival_function(ax=ax_km, ci_show=True, color="green")
        kmf_d.plot_survival_function(ax=ax_km, ci_show=True, color="red")
        ax_km.set_title(f"Line {line + 1}")
        ax_km.set_ylabel("Survival probability")

        tc, rc = _rmst_from_km(kmf_c)
        td, rd = _rmst_from_km(kmf_d)
        ax_rmst.plot(tc, rc, color="green", label="Matched")
        ax_rmst.plot(td, rd, color="red", label="Differed")
        ax_rmst.set_xlabel("Time")
        ax_rmst.set_ylabel("RMST(t)")
        ax_km.legend()
        ax_rmst.legend()
        ax_km.set_xlim(0, xmax)
        ax_rmst.set_xlim(0, xmax)

    fig.suptitle("Model recommendation vs. actual treatment")
    plt.tight_layout()
    plt.show()


conc, disc, recommended_treatment, observed_treatment, trial_mask = run_ai_vs_doctor(
    test_loader, interval_bounds=interval_bounds, eval_time=TRIAL_HORIZON
)
plot_conc_disc(conc, disc)

In [ ]:
def plot_treatment_mix(recommended, observed, treatment_dict, mask):
    n_lines = recommended.shape[1]
    rec = recommended.cpu().numpy()
    obs = observed.cpu().numpy()
    m = mask.bool().cpu().numpy()

    fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))
    cmap = plt.colormaps["tab20"]
    uniq = np.unique(np.concatenate([rec.ravel(), obs.ravel()]))
    colors = {int(t): cmap(i % 20) for i, t in enumerate(uniq)}

    def _plot(data, axis, title):
        used = set()
        for line in range(n_lines):
            ml = m[:, line]
            values, counts = np.unique(data[ml, line], return_counts=True)
            order = np.argsort(counts)
            bottom = 0
            for val, cnt in zip(values[order], counts[order]):
                val = int(val)
                label = treatment_dict.get(val) if val not in used else None
                used.add(val)
                axis.bar(line, cnt, bottom=bottom, color=colors[val], label=label)
                bottom += cnt
        axis.set_xticks(range(n_lines))
        axis.set_xticklabels([f"Line {i + 1}" for i in range(n_lines)])
        axis.set_ylabel("Count")
        axis.set_title(title)

    _plot(rec, ax[0], "Recommended treatment mix")
    _plot(obs, ax[1], "Observed treatment mix")
    ax[1].legend(loc="upper right", bbox_to_anchor=(1.35, 1.0), fontsize=8)
    plt.tight_layout()
    plt.show()


plot_treatment_mix(
    recommended_treatment, observed_treatment, data_module.treatment_dict, trial_mask
)